In [ ]:
import numpy as np
import os
from astropy.io import fits
from astropy.table import Table
import xarray as xr
import pandas as pd

In [ ]:
 



# ============================================================
# Helper function
# ============================================================

def load_fits_table(path):
    """
    Load a FITS binary table into a pandas DataFrame
    and replace dots in column names with underscores.
    """
    
    # Read FITS table
    table = Table(fits.open(path)[1].data)

    # Convert to pandas
    df = table.to_pandas()

    # Clean column names
    df.columns = [col.replace('.', '_') for col in df.columns]

    # Use galaxy ID as index
    df = df.set_index('id')

    return df


# ============================================================
# Load datasets
# ============================================================

obs = load_fits_table(
    '/project/galaxies/tjuchau/projects/CLOUDY_scripts/Galaxy_class_project3/run_AGN/out/observations.fits'
)

noagn = load_fits_table(
    '/project/galaxies/tjuchau/projects/CLOUDY_scripts/Galaxy_class_project3/run_noAGN/out/results.fits'
)

agn = load_fits_table(
    '/project/galaxies/tjuchau/projects/CLOUDY_scripts/Galaxy_class_project3/run_AGN/out/results.fits'
)


# ============================================================
# Convert to xarray datasets
# ============================================================

ds_obs = xr.Dataset.from_dataframe(obs).expand_dims(model=['obs'])

ds_noagn = xr.Dataset.from_dataframe(noagn).expand_dims(model=['noAGN'])

ds_agn = xr.Dataset.from_dataframe(agn).expand_dims(model=['AGN'])


# ============================================================
# Combine into one Dataset
# ============================================================

ds = xr.concat(
    [ds_obs, ds_noagn, ds_agn],
    dim='model'
)


# ============================================================
# Optional cleanup
# ============================================================

# Rename dimension from "id" -> "galaxy"
ds = ds.rename({'id': 'galaxy'})


In [ ]:
ds

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

# ------------------------------------------------------------
# Select galaxies and variables
# ------------------------------------------------------------

gal_names = ['NGC4151', 'NGC1068', 'NGC0628', 'NGC1097', 'NGC5055']

keep = [
    'best_stellar_m_star',
    'best_reduced_chi_square',
    'bayes_agn_fracAGN',
    'best_attenuation_generic_bessell_V',
    'best_dust_luminosity',
    'best_sfh_sfr'
]

# ------------------------------------------------------------
# Subset dataset
# ------------------------------------------------------------

sub = ds[keep].sel(galaxy=gal_names)

# ------------------------------------------------------------
# Convert to dataframe
# ------------------------------------------------------------

df = sub.to_dataframe().reset_index()

# ------------------------------------------------------------
# Keep only AGN + noAGN models
# ------------------------------------------------------------

df = df[df['model'].isin(['AGN', 'noAGN'])]

# ------------------------------------------------------------
# Rename columns for prettier LaTeX output
# ------------------------------------------------------------

df = df.rename(columns={
    'galaxy': 'Galaxy',
    'model': 'Model',
    'best_reduced_chi_square': r'Reduced $\chi^2$',
    'best_stellar_m_star': r'$M_\star$ $(M_\odot)$',
    'best_sfh_sfr': r'SFR $(M_\odot\,yr^{-1})$',
    'best_attenuation_generic_bessell_V': r'$A_V$',
    'best_dust_luminosity': r'$L_{\rm dust}$ $(L_\odot)$',
    'bayes_agn_fracAGN': 'AGN Fraction'
})

# ------------------------------------------------------------
# Replace NaNs in AGN fraction column with em-dash
# ------------------------------------------------------------

df['AGN Fraction'] = df['AGN Fraction'].replace(np.nan, '---')

# ------------------------------------------------------------
# Optional formatting
# ------------------------------------------------------------

# Scientific notation for masses/luminosities
df[r'$M_\star$ $(M_\odot)$'] = (
    df[r'$M_\star$ $(M_\odot)$']
    .map(lambda x: f'{x:.3e}')
)

df[r'$L_{\rm dust}$ $(L_\odot)$'] = (
    (df[r'$L_{\rm dust}$ $(L_\odot)$']/l_sun)
    .map(lambda x: f'{x:.3e}')
)

# Decimal formatting
df[r'Reduced $\chi^2$'] = (
    df[r'Reduced $\chi^2$']
    .map(lambda x: f'{x:.3f}')
)

df[r'SFR $(M_\odot\,yr^{-1})$'] = (
    df[r'SFR $(M_\odot\,yr^{-1})$']
    .map(lambda x: f'{x:.3f}')
)

df[r'$A_V$'] = (
    df[r'$A_V$']
    .map(lambda x: f'{x:.3f}')
)

# ------------------------------------------------------------
# Reorder columns
# ------------------------------------------------------------

df = df[[
    'Galaxy',
    'Model',
    r'Reduced $\chi^2$',
    r'$M_\star$ $(M_\odot)$',
    r'SFR $(M_\odot\,yr^{-1})$',
    r'$A_V$',
    r'$L_{\rm dust}$ $(L_\odot)$',
    'AGN Fraction'
]]

# ------------------------------------------------------------
# Convert to LaTeX
# ------------------------------------------------------------

latex_table = df.to_latex(
    index=False,
    escape=False,
    column_format='llcccccc'
)

print(latex_table)

In [ ]:
gal_names = ['NGC4151', 'NGC1068', 'NGC0628', 'NGC1097','NGC5055']
keep = ['best_stellar_m_star', 'best_reduced_chi_square', 'bayes_agn_fracAGN', "best_attenuation_generic_bessell_V", 'best_dust_luminosity', 'best_sfh_sfr']
ds['bayes_agn_fracAGN']


In [ ]:

l_sun = 3.828e+26
x =pd.DataFrame(ds['best_dust_luminosity']/l_sun)
x.to_latex()

In [ ]:
ds